In [1]:
import os
print(os.listdir("/kaggle/input/datasets/hearfool/vggface2"))

['val', 'train']


In [2]:
import os
import shutil
import random
from pathlib import Path

src = "/kaggle/input/datasets/hearfool/vggface2/train"
dst = "/kaggle/working/faces_by_identity_vgg"

N_IDENTITIES      = 3000
MIN_IMAGES        = 20
MAX_IMAGES_PER_ID = 60

random.seed(42)
os.makedirs(dst, exist_ok=True)

# VGGFace2 đã chia sẵn theo folder-per-identity (n000002/, n000003/...)
# => KHÔNG tách identity từ filename như Korean Family nữa.
all_ids = sorted(os.listdir(src))
eligible = [
    pid for pid in all_ids
    if os.path.isdir(os.path.join(src, pid))
    and len(os.listdir(os.path.join(src, pid))) >= MIN_IMAGES
]
print(f"Identity đủ điều kiện (>= {MIN_IMAGES} ảnh): {len(eligible)}")

random.shuffle(eligible)
selected = eligible[:N_IDENTITIES]

for pid in selected:
    src_dir = os.path.join(src, pid)
    dst_dir = os.path.join(dst, pid)
    os.makedirs(dst_dir, exist_ok=True)

    imgs = [f for f in os.listdir(src_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
    random.shuffle(imgs)
    for img in imgs[:MAX_IMAGES_PER_ID]:
        shutil.copy(os.path.join(src_dir, img), os.path.join(dst_dir, img))

print(f"Done converting dataset — đã chọn {len(selected)} identity → {dst}")

Identity đủ điều kiện (>= 20 ảnh): 480
Done converting dataset — đã chọn 300 identity → /kaggle/working/faces_by_identity_vgg


In [3]:
root = "/kaggle/working/faces_by_identity_vgg"

counts = {}

for person in os.listdir(root):
    p = os.path.join(root, person)
    if os.path.isdir(p):
        counts[person] = len(os.listdir(p))

print("num identities:", len(counts))
print("min:", min(counts.values()))
print("max:", max(counts.values()))

num identities: 300
min: 60
max: 60


In [4]:
import os, math, json, copy, random
from dataclasses import dataclass, asdict, field
from typing import Dict, List, Tuple, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets, models, transforms
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

In [5]:
# Cell 3: Convert train_images -> faces_by_identity
src = "/kaggle/input/datasets/hearfool/vggface2/train"
dst = "/kaggle/working/faces_by_identity_vgg"

os.makedirs(dst, exist_ok=True)

num_copied = 0
for img in os.listdir(src):
    if not img.lower().endswith((".jpg", ".jpeg", ".png")):
        continue

    identity = img.split("_")[0]   # ví dụ: F0851
    identity_dir = os.path.join(dst, identity)
    os.makedirs(identity_dir, exist_ok=True)

    shutil.copy(
        os.path.join(src, img),
        os.path.join(identity_dir, img)
    )
    num_copied += 1

print("Done converting dataset")
print("Total copied:", num_copied)


Done converting dataset
Total copied: 0


In [6]:
# Cell 4: Kiểm tra nhanh dataset đã convert
root = "/kaggle/working/faces_by_identity_vgg"
counts = {}

for person in os.listdir(root):
    p = os.path.join(root, person)
    if os.path.isdir(p):
        counts[person] = len([
            f for f in os.listdir(p)
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
        ])

print("num identities:", len(counts))
print("min:", min(counts.values()) if counts else 0)
print("max:", max(counts.values()) if counts else 0)
print("sample:", list(counts.items())[:10])


num identities: 300
min: 60
max: 60
sample: [('n000108', 60), ('n000166', 60), ('n000073', 60), ('n000298', 60), ('n000042', 60), ('n000176', 60), ('n000491', 60), ('n000327', 60), ('n000112', 60), ('n000236', 60)]


In [7]:
import os
cache = "./runs/lpeu_v7_vggface2/original_model.pt"
if os.path.exists(cache):
    os.remove(cache)
    print("Cache deleted")
else:
    print("No cache — will train fresh")

No cache — will train fresh


In [8]:
# ===========================================================================
# ▓  CELL B: CONFIG
# ===========================================================================

@dataclass
class Config:
    # ── paths ─────────────────────────────────────────────────────────────
    dataset_root:  str = "./faces_by_identity_vgg"
    output_dir:    str = "./runs/lpeu_v7_vggface2"
    seed:          int = 42
    device:        str = "cuda" if torch.cuda.is_available() else "cpu"

    # ── data ──────────────────────────────────────────────────────────────
    image_size:               int   = 128
    batch_size:               int   = 32
    num_workers:              int   = 2
    min_images_per_identity:  int   = 8
    train_ratio:              float = 0.70
    val_ratio:                float = 0.10
    test_ratio:               float = 0.20

    # ── model ─────────────────────────────────────────────────────────────
    embedding_dim:       int   = 128
    pretrained_backbone: bool  = True
    arcface_s:           float = 32.0
    arcface_m:           float = 0.30

    # ── original training ─────────────────────────────────────────────────
    # v7.1: log thực tế cho thấy val_acc vẫn tăng ở epoch 50/50 (chưa hội tụ)
    # → nâng epoch trần + early-stop theo patience thay vì cắt cứng ở 50
    original_epochs:      int   = 120
    original_lr:          float = 0.05
    original_momentum:    float = 0.9
    original_wd:          float = 1e-4
    warmup_epochs:        int   = 5
    early_stop_patience:  int   = 20   # dừng nếu val_acc không cải thiện sau N epoch
    lr_reduce_patience:   int   = 4 

    # ── dataset splits ────────────────────────────────────────────────────
    num_forget_ids: int = 100
    neighbor_k:     int = 64

    # ── staged unlearning schedule ─────────────────────────────────────────
    stage1_epochs: int = 1    # Phase 1: anchor OFF
    stage2_epochs: int = 14   # Phase 2: anchor 50%
    stage3_epochs: int = 5    # Phase 3: anchor 100%

    # ── LR ────────────────────────────────────────────────────────────────
    unlearn_lr:      float = 5e-6
    unlearn_wd:      float = 1e-6
    layer4_lr_mult:  float = 0.01

    # v7.2 FIX — manual step size cho bước FORGET (neck + layer4).
    # Log thực tế: dù pcgrad_boost=40×, erase/repel/proto_sim gần như KHÔNG
    # đổi suốt 20 epoch (proto_sim 0.9994→0.9955, drop tổng cộng chỉ +0.0045).
    # Nguyên nhân: bước forget đang dùng chung optimizer AdamW với bước retain.
    # AdamW chuẩn hoá gradient theo m/√v (running RMS) — nhân gradient với một
    # hằng số k (boost) làm m và v cùng scale theo k và k², nên tỉ lệ m/√v gần
    # như KHÔNG đổi. Nói cách khác: boost×40 gần như vô tác dụng khi đi qua
    # AdamW. Fix: bước forget áp dụng SGD thủ công (param -= lr*grad) trực
    # tiếp lên neck/layer4 SAU khi PCGrad chiếu + nhân boost, bỏ qua AdamW cho
    # đúng 2 nhóm tham số này (ArcFace vẫn update qua AdamW như cũ).
    forget_manual_lr: float = 2e-4

    # ── retain path ───────────────────────────────────────────────────────
    w_kd_global:    float = 30.0
    kd_temperature: float = 2.0
    k_anchor:       int   = 30    # tăng từ 10 — bao phủ wider region với 10 forget IDs
    w_kd_local:     float = 15.0
    use_ewc:        bool  = True
    w_ewc:          float = 50.0
    w_ce_retain:    float = 5.0   # tăng từ 2 — CE mạnh hơn trong repair

    # ── RETAIN-PROTOTYPE ANCHOR (mới — mở rộng ý tưởng prototype của LPEU) ──
    # Đối xứng với L_erase (đẩy forget ra xa prototype của chính nó):
    # ở đây kéo MỖI mẫu retain về lại prototype (đóng băng, tính trước khi
    # unlearn) của ĐÚNG lớp của nó. Vẫn là "local prototype" — chỉ khác là
    # dùng để NEO thay vì XOÁ. Giúp bảo vệ trực tiếp độ chính xác phân loại
    # (ArcFace về cơ bản so cosine với vector lớp, prototype gần hướng đó).
    use_proto_retain: bool  = True
    w_proto_retain:   float = 20.0

    # ── IN-LOOP REPAIR ────────────────────────────────────────────────────
    use_inloop_repair: bool  = True
    w_repair:          float = 20.0

    # ── POST-UNLEARNING REPAIR (2-phase redesign) ─────────────────────────
    # Phase R1: neck + retain arcface cùng adapt với embedding space mới
    # Phase R2: KD stabilization để không drift quá xa
    # KEY FIX: chỉ freeze FORGET arcface, KHÔNG restore retain arcface về original
    # → tránh neck-arcface mismatch gây retain_acc < original
    use_post_repair:  bool  = True
    repair_epochs:    int   = 20    # tăng từ 5 — đủ thời gian để adapt
    repair_lr:        float = 1e-5  # tăng mạnh từ 2e-6 — neck cần adapt tích cực hơn

    repair_layer4_lr_mult: float = 0.25 
    repair_r2_kd_mult: float = 1.0

    # ── forget path ───────────────────────────────────────────────────────
    w_erase:       float = 2.0
    w_kl_uniform:  float = 5.0    # tăng từ 3 — entropy tăng mạnh hơn
    w_repel:       float = 4.0    # giảm từ 8 — bớt collateral damage
    repel_margin:  float = -0.3

    # ── PCGrad ────────────────────────────────────────────────────────────
    use_pcgrad:   bool  = True
    pcgrad_boost: float = 40.0

    # ── clipping ──────────────────────────────────────────────────────────
    grad_clip_retain: float = 0.1
    grad_clip_forget: float = 2.0

    # ── safety ────────────────────────────────────────────────────────────
    retain_acc_floor: float = 0.90  # tăng từ 0.85 — safety stop sớm hơn
    # v7.1: log thực tế cho thấy safety-stop có thể trigger ngay epoch thứ 2
    # (stage1_epochs=1) và revert model về gần như nguyên bản chưa unlearn.
    # → thêm grace period + đếm số epoch VI PHẠM LIÊN TIẾP trước khi thực sự
    #   dừng, để cơ chế in-loop repair/EWC có cơ hội kéo retain_acc hồi phục.
    safety_grace_epochs: int = 3   # không check an toàn trước epoch này
    safety_patience:     int = 2   # phải vi phạm floor liên tiếp N epoch mới dừng

    # ── MIA ───────────────────────────────────────────────────────────────
    mia_n_members:    int = 300
    mia_n_nonmembers: int = 300

    unlearn_epochs: int = 0   # tính tự động trong __post_init__

    def __post_init__(self):
        self.unlearn_epochs = self.stage1_epochs + self.stage2_epochs + self.stage3_epochs


cfg = Config()
os.makedirs(cfg.output_dir, exist_ok=True)

def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(cfg.seed)

print(f"{'─'*65}")
print(f"  LPEU-v7  |  Device: {cfg.device}")
print(f"{'─'*65}")
print(f"  Model:  emb_dim={cfg.embedding_dim}  s={cfg.arcface_s}  m={cfg.arcface_m}  [Simple Neck]")
print(f"  Staged unlearning: Phase1={cfg.stage1_epochs}e / Phase2={cfg.stage2_epochs}e / Phase3={cfg.stage3_epochs}e")
print(f"  EWC:    {'ON' if cfg.use_ewc else 'OFF'}  w_ewc={cfg.w_ewc}")
print(f"  Forget: w_erase={cfg.w_erase}  w_repel={cfg.w_repel}  margin={cfg.repel_margin}  boost={cfg.pcgrad_boost}×")
print(f"  Forget manual LR (v7.2, bypass AdamW): {cfg.forget_manual_lr:.1e}")
print(f"{'─'*65}")

─────────────────────────────────────────────────────────────────
  LPEU-v7  |  Device: cuda
─────────────────────────────────────────────────────────────────
  Model:  emb_dim=128  s=32.0  m=0.3  [Simple Neck]
  Staged unlearning: Phase1=1e / Phase2=14e / Phase3=5e
  EWC:    ON  w_ewc=50.0
  Forget: w_erase=2.0  w_repel=4.0  margin=-0.3  boost=40.0×
  Forget manual LR (v7.2, bypass AdamW): 2.0e-04
─────────────────────────────────────────────────────────────────


In [9]:
# ===========================================================================
# ▓  CELL C: DATASET UTILITIES
# ===========================================================================

class IndexedSubset(Dataset):
    """Wraps a dataset; returns (x, y, original_idx)."""
    def __init__(self, dataset, indices):
        self.dataset = dataset
        self.indices = list(indices)
    def __len__(self):  return len(self.indices)
    def __getitem__(self, i):
        x, y = self.dataset[self.indices[i]]
        return x, y, self.indices[i]


def build_transforms(image_size: int):
    train_tf = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(0.2, 0.2, 0.2),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
    ])
    eval_tf = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
    ])
    return train_tf, eval_tf


def load_full_dataset(cfg: Config):
    train_tf, eval_tf = build_transforms(cfg.image_size)
    base = datasets.ImageFolder(cfg.dataset_root)
    class_to_indices: Dict[int, List[int]] = {}
    for idx, (_, y) in enumerate(base.samples):
        class_to_indices.setdefault(y, []).append(idx)
    valid_classes = {c for c, idxs in class_to_indices.items()
                     if len(idxs) >= cfg.min_images_per_identity}
    filtered = [i for i, (_, y) in enumerate(base.samples) if y in valid_classes]
    train_base = datasets.ImageFolder(cfg.dataset_root, transform=train_tf)
    eval_base  = datasets.ImageFolder(cfg.dataset_root, transform=eval_tf)
    return train_base, eval_base, filtered, sorted(valid_classes)


def split_by_identity(dataset, filtered_indices, cfg: Config):
    class_to_indices: Dict[int, List[int]] = {}
    for idx in filtered_indices:
        _, y = dataset.samples[idx]
        class_to_indices.setdefault(y, []).append(idx)

    train_idx, val_idx, test_idx = [], [], []
    eligible_forget = []

    for cls, idxs in class_to_indices.items():
        random.shuffle(idxs)
        n = len(idxs)
        n_tr = max(1, int(n * cfg.train_ratio))
        n_va = max(1, int(n * cfg.val_ratio))
        n_te = max(1, n - n_tr - n_va)
        if n_tr + n_va + n_te > n:
            n_tr = max(1, n_tr - 1)
        train_idx.extend(idxs[:n_tr])
        val_idx.extend(idxs[n_tr:n_tr+n_va])
        test_idx.extend(idxs[n_tr+n_va:n_tr+n_va+n_te])
        if n_tr >= 2: eligible_forget.append(cls)

    forget_ids = random.sample(eligible_forget, cfg.num_forget_ids)
    fset = set(forget_ids)

    forget_idx, retain_train_idx = [], []
    for idx in train_idx:
        _, y = dataset.samples[idx]
        (forget_idx if y in fset else retain_train_idx).append(idx)

    retain_test_idx, forget_test_idx = [], []
    for idx in test_idx:
        _, y = dataset.samples[idx]
        (forget_test_idx if y in fset else retain_test_idx).append(idx)

    # v7.3 — retain-only VAL split (mirror retain_test_idx logic nhưng trên
    # val_idx). Dùng để best-checkpoint-select trong run_retain_repair mà
    # KHÔNG "nhìn" vào retain_test_idx (tránh leak test set vào lựa chọn
    # checkpoint — xem lý do ở run_retain_repair).
    retain_val_idx, forget_val_idx = [], []
    for idx in val_idx:
        _, y = dataset.samples[idx]
        (forget_val_idx if y in fset else retain_val_idx).append(idx)

    return dict(
        train_idx=train_idx, val_idx=val_idx, test_idx=test_idx,
        forget_idx=forget_idx, retain_train_idx=retain_train_idx,
        retain_test_idx=retain_test_idx, forget_test_idx=forget_test_idx,
        retain_val_idx=retain_val_idx, forget_val_idx=forget_val_idx,
        forget_ids=forget_ids,
    )


def make_loader(dataset, batch_size, shuffle, num_workers=2):
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,
                      num_workers=num_workers, pin_memory=True)

In [10]:
# ===========================================================================
# ▓  CELL D: MODEL ARCHITECTURE — v7 (Simple Neck)
# ===========================================================================

class ArcMarginProduct(nn.Module):
    """ArcFace margin loss head. Returns cosine-scaled logits."""
    def __init__(self, in_features, out_features, s=32.0, m=0.30):
        super().__init__()
        self.s, self.m = s, m
        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)
        self.cos_m = math.cos(m); self.sin_m = math.sin(m)
        self.th    = math.cos(math.pi - m)
        self.mm    = math.sin(math.pi - m) * m

    def forward(self, emb, labels=None):
        emb = F.normalize(emb, dim=1)
        W   = F.normalize(self.weight, dim=1)
        cos = F.linear(emb, W)            # [B, C]
        if labels is None:
            return cos * self.s
        sin  = torch.sqrt((1.0 - cos**2).clamp(1e-7))
        phi  = cos * self.cos_m - sin * self.sin_m
        phi  = torch.where(cos > self.th, phi, cos - self.mm)
        oh   = torch.zeros_like(cos)
        oh.scatter_(1, labels.view(-1,1), 1.0)
        return (oh * phi + (1.0 - oh) * cos) * self.s


class FaceModel(nn.Module):
    """
    ResNet-18 + Simple Linear Neck + ArcFace.

    KEY CHANGE vs v6:
    -----------------
    v6 used BN-Neck (BN→Linear→BN) which caused embedding collapse:
    all same-identity embeddings converged to near-identical points
    (intra_cluster_sim = 0.905), making pair repulsion and anti-prototype
    push unable to scatter the cluster.

    v7 uses a single Linear layer (no BatchNorm):
    - Embeddings maintain natural inter-sample diversity
    - Expected intra_cluster_sim ≈ 0.35–0.50 (same as v5's working model)
    - Pair repulsion can now function properly

    forward(x, labels) → (embedding [B, D], logits [B, C])
    """
    def __init__(self, num_classes: int, embedding_dim: int = 128,
                 pretrained: bool = True, s: float = 32.0, m: float = 0.30):
        super().__init__()
        bb = models.resnet18(
            weights=models.ResNet18_Weights.DEFAULT if pretrained else None
        )
        in_feat = bb.fc.in_features    # 512 for ResNet-18
        bb.fc   = nn.Identity()
        self.backbone      = bb
        self.embedding_dim = embedding_dim

        # ★ Simple neck — single Linear, no BN
        self.neck = nn.Linear(in_feat, embedding_dim, bias=False)

        self.arcface = ArcMarginProduct(embedding_dim, num_classes, s=s, m=m)

    def forward(self, x, labels=None):
        feat   = self.backbone(x)                # [B, 512]
        emb    = self.neck(feat)                  # [B, D]
        emb    = F.normalize(emb, dim=1)         # unit sphere
        logits = self.arcface(emb, labels)
        return emb, logits


def clone_model(model: nn.Module) -> nn.Module:
    return copy.deepcopy(model)

In [11]:
# ===========================================================================
# ▓  CELL E: ORIGINAL MODEL TRAINING
# ===========================================================================

def train_original_model(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    cfg: Config,
) -> nn.Module:
    """
    v7.2 FIX — bỏ cosine LR schedule cố định theo original_epochs.

    Vấn đề phát hiện từ log thực tế: sau khi tăng original_epochs 50→120 (v7.1),
    retain_accuracy của Original model TỆ HƠN (11.35%→5.99%) và forget_accuracy
    (đo trên train-images bị nhớ) lại TĂNG (26%→31.4%) — dấu hiệu điển hình của
    overfit cục bộ + generalize kém, KHÔNG phải do model "chưa học đủ".

    Root cause: cosine schedule dùng T = original_epochs − warmup_epochs làm
    mẫu số. Khi original_epochs tăng 50→120, T tăng 45→115 → ở CÙNG một epoch
    (ví dụ epoch 45), LR còn lại cao hơn nhiều so với schedule cũ (schedule cũ
    gần như anneal về 0 ở epoch 45–50, schedule mới mới đi được ~1/3 chặng
    đường) → optimizer chưa kịp "hạ nhiệt", dễ overfit theo từng batch cụ thể
    (kể cả 242 ảnh forget-train) thay vì hội tụ tổng quát.

    Fix: warmup tuyến tính như cũ, sau đó giảm LR thủ công theo CÙNG MỘT
    best_val dùng cho early-stopping (không phải bằng ReduceLROnPlateau).

    v7.3 FIX — BUG PHÁT HIỆN TỪ LOG THỰC TẾ (VGGFace2):
    ReduceLROnPlateau tự theo dõi "best" nội bộ RIÊNG, tách biệt khỏi biến
    best_val ở đây. Vì plateau_scheduler.step() chỉ bắt đầu được gọi từ sau
    warmup (epoch >= warmup_epochs), nếu đỉnh val_acc thật (vd 0.1272) đã đạt
    được TRONG lúc warmup, thì giá trị val_acc thấp đầu tiên sau warmup (vd
    0.046) trở thành "best" nội bộ của scheduler. Nếu sau đó val_acc bò dần
    lên (0.046→0.057→0.069) — vẫn thấp hơn NHIỀU so với đỉnh thật — scheduler
    vẫn coi đó là "đang cải thiện" so với mốc nội bộ của nó và KHÔNG BAO GIỜ
    đếm đủ patience để giảm LR, trong khi early-stop (so với đỉnh thật) đúng
    là không cải thiện nên đếm đủ patience và dừng training trước khi LR kịp
    giảm lần nào. Hệ quả quan sát được: model train nguyên 21 epoch ở LR gốc
    (0.05, quá cao) không đổi → không bao giờ hội tụ lại quanh đỉnh đã đạt →
    retain_accuracy cuối cùng (13.6%) thấp hơn nhiều so với đỉnh thật (12.72%
    đã từng đạt được) → cụm identity chưa tách biệt rõ → toàn bộ downstream
    (erase/repel, cluster metrics, MIA AUC) bị ảnh hưởng.
    Fix: dùng chung DUY NHẤT một best_val cho cả early-stop và giảm LR.
    """
    device = cfg.device
    model.to(device)
    optimizer = torch.optim.SGD(
        model.parameters(), lr=cfg.original_lr,
        momentum=cfg.original_momentum, weight_decay=cfg.original_wd,
        nesterov=True,
    )
    min_lr = cfg.original_lr * 1e-3
    ce = nn.CrossEntropyLoss()
    best_val, best_state = -1.0, None
    epochs_no_improve = 0

    for epoch in range(cfg.original_epochs):
        # Warmup tuyến tính (giữ nguyên như bản gốc)
        if epoch < cfg.warmup_epochs:
            warmup_lr = cfg.original_lr * (epoch + 1) / cfg.warmup_epochs
            for g in optimizer.param_groups:
                g['lr'] = warmup_lr

        model.train(); total_loss = 0.0
        for x, y, _ in train_loader:
            x, y = x.to(device), y.to(device)
            _, logits = model(x, y)
            loss = ce(logits, y)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += float(loss)

        val_acc = evaluate_accuracy(model, val_loader, device)

        if val_acc > best_val:
            best_val = val_acc; best_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            # Giảm LR dựa trên ĐÚNG best_val dùng cho early-stop — không còn
            # trạng thái "best" thứ hai nào có thể lệch pha nữa.
            if (epoch >= cfg.warmup_epochs
                    and epochs_no_improve % cfg.lr_reduce_patience == 0):
                for g in optimizer.param_groups:
                    new_lr = max(g['lr'] * 0.5, min_lr)
                    g['lr'] = new_lr
                print(f"[Train]   ↓ LR giảm còn {new_lr:.2e} "
                      f"(no_improve={epochs_no_improve}, best_val={best_val:.4f})")

        cur_lr = optimizer.param_groups[0]['lr']
        if (epoch + 1) % 5 == 0 or epoch == cfg.original_epochs - 1:
            print(f"[Train][{epoch+1:02d}/{cfg.original_epochs}]  "
                  f"loss={total_loss/len(train_loader):.4f}  val_acc={val_acc:.4f}  "
                  f"best_val={best_val:.4f}  no_improve={epochs_no_improve}  lr={cur_lr:.2e}")
        # early-stop theo patience — giờ đi cùng cơ chế giảm LR thủ công ở trên
        # (cùng best_val) nên khi LR đã giảm sâu mà val_acc vẫn không nhích,
        # dừng lại là hợp lý (đã hội tụ).
        if epochs_no_improve >= cfg.early_stop_patience:
            print(f"[Train] Early stop tại epoch {epoch+1}: "
                  f"val_acc không cải thiện sau {cfg.early_stop_patience} epoch "
                  f"(best_val={best_val:.4f}, lr={cur_lr:.2e})")
            break
    model.load_state_dict(best_state)
    return model

In [12]:
# ===========================================================================
# ▓  CELL F: PROTOTYPE & K-NN UTILITIES
# ===========================================================================

@torch.no_grad()
def compute_class_prototypes(
    model: nn.Module,
    loader: DataLoader,
    num_classes: int,
    device: str,
) -> torch.Tensor:
    """Mean unit-norm embedding per class. Returns [num_classes, D] on CPU."""
    model.eval()
    D = model.embedding_dim
    sums   = torch.zeros(num_classes, D)
    counts = torch.zeros(num_classes)
    for x, y, _ in loader:
        z, _ = model(x.to(device), None)
        z = z.cpu()
        for i, cls in enumerate(y.tolist()):
            sums[cls] += z[i]
            counts[cls] += 1
    nonzero = counts > 0
    sums[nonzero] = F.normalize(sums[nonzero], dim=1)
    return sums


@torch.no_grad()
def compute_forget_prototype(
    model: nn.Module,
    forget_loader: DataLoader,
    device: str,
) -> torch.Tensor:
    """Unit-norm mean embedding of forget-set samples. Returns CPU tensor."""
    model.eval()
    embs = [model(x.to(device), None)[0].cpu() for x, _, _ in forget_loader]
    z = torch.cat(embs)
    return F.normalize(z.mean(0, keepdim=True), dim=1).squeeze(0)


def find_knn_retain_classes(
    forget_proto: torch.Tensor,       # [D] CPU
    all_prototypes: torch.Tensor,     # [C, D] CPU
    forget_ids: List[int],
    k: int,
) -> List[int]:
    """K retain classes whose prototype is cosine-closest to forget_proto."""
    fp = F.normalize(forget_proto.unsqueeze(0), dim=1)
    sims = (all_prototypes @ fp.T).squeeze(1)
    for fid in forget_ids:
        sims[fid] = -2.0
    topk = torch.topk(sims, k=min(k, int((sims > -1.5).sum()))).indices.tolist()
    return topk


def build_local_anchor_loader(
    eval_base: datasets.ImageFolder,
    retain_train_idx: List[int],
    anchor_class_ids: List[int],
    batch_size: int,
    num_workers: int,
) -> Optional[DataLoader]:
    """DataLoader with only K anchor-class samples."""
    anchor_set = set(anchor_class_ids)
    anchor_indices = [
        idx for idx in retain_train_idx
        if eval_base.samples[idx][1] in anchor_set
    ]
    if not anchor_indices:
        print("[v7] ⚠ No anchor samples found.")
        return None
    ds = IndexedSubset(eval_base, anchor_indices)
    print(f"[v7] Anchor loader: {len(anchor_class_ids)} classes, {len(anchor_indices)} samples")
    return DataLoader(ds, batch_size=batch_size, shuffle=True,
                      num_workers=num_workers, pin_memory=True)

In [13]:
# ===========================================================================
# ▓  CELL G: EWC UTILITIES (NEW IN v7)
# ===========================================================================

@torch.no_grad()
def snapshot_neck_params(model: nn.Module) -> Dict[str, torch.Tensor]:
    """Save a copy of neck (and layer4) parameter values as θ* for EWC."""
    snap = {}
    for name, param in model.named_parameters():
        if "neck" in name or "backbone.layer4" in name:
            snap[name] = param.data.clone().cpu()
    return snap


def compute_ewc_fisher(
    model: nn.Module,
    retain_loader: DataLoader,
    device: str,
    n_batches: int = 30,
) -> Dict[str, torch.Tensor]:
    """
    Compute diagonal Fisher Information on retain set for neck + layer4 params.

    Fisher_i = E[( ∂log p(y|x) / ∂θ_i )²]

    Estimated by:
      F_i = (1/N) Σ_{x,y ∈ D_r} (∂CE/∂θ_i)²

    This tells us how important each neck weight is for retain accuracy.
    High F_i → θ_i must not change → heavy EWC penalty on that weight.
    """
    model.eval()
    fisher: Dict[str, torch.Tensor] = {}
    target_names = set()
    for name, param in model.named_parameters():
        if "neck" in name or "backbone.layer4" in name:
            fisher[name] = torch.zeros_like(param.data)
            target_names.add(name)

    ce = nn.CrossEntropyLoss()
    count = 0
    for x, y, _ in retain_loader:
        if count >= n_batches:
            break
        x, y = x.to(device), y.to(device)
        model.zero_grad()
        _, logits = model(x, y)
        loss = ce(logits, y)
        loss.backward()
        for name, param in model.named_parameters():
            if name in target_names and param.grad is not None:
                fisher[name] += param.grad.data.pow(2)
        count += 1

    for name in fisher:
        fisher[name] /= max(count, 1)

    total_params = sum(v.numel() for v in fisher.values())
    mean_f = sum(float(v.mean()) for v in fisher.values()) / max(len(fisher), 1)
    print(f"[v7] EWC Fisher computed: {len(fisher)} param groups, "
          f"{total_params:,} params, mean_F={mean_f:.6f}")
    return fisher


def compute_ewc_loss(
    model: nn.Module,
    theta_star: Dict[str, torch.Tensor],
    fisher: Dict[str, torch.Tensor],
    w_ewc: float,
    device: str,
) -> torch.Tensor:
    """
    EWC penalty: L_ewc = (w_ewc/2) × Σ_i F_i × (θ_i − θ*_i)²

    When θ deviates from θ* on a high-Fisher weight (important for retain),
    this penalty is large → neck update is constrained.
    When θ stays close to θ* (retain behaviour preserved), penalty ≈ 0.
    """
    loss = torch.tensor(0.0, device=device, requires_grad=True)
    for name, param in model.named_parameters():
        if name in fisher and name in theta_star:
            F_i      = fisher[name].to(device)
            theta_i  = theta_star[name].to(device)
            loss = loss + (F_i * (param - theta_i).pow(2)).sum()
    return (w_ewc / 2.0) * loss


In [14]:
# ===========================================================================
# ▓  CELL H: LOSS FUNCTIONS (v7)
# ===========================================================================

class ForgetLossV7(nn.Module):
    """
    Three-component forget loss (same as v6 but with aggressive repel_margin).

    Key change: repel_margin = -0.30 (was 0.0 in v6).
    This forces pair repulsion to push forget embeddings until
    cos(z_fi, z_fj) < -0.3 — ensuring true scatter, not just decorrelation.
    """
    def __init__(self, cfg: Config, num_classes: int):
        super().__init__()
        self.cfg   = cfg
        self.C     = num_classes
        self.log_C = math.log(num_classes)

    def forward(
        self,
        z_f:       torch.Tensor,   # [B, D] unit-norm forget embeddings
        old_proto: torch.Tensor,   # [D]    frozen prototype (on device)
        logits_f:  torch.Tensor,   # [B, C] ArcFace logits
        w_erase:   float = None,   # override for staged schedule
        w_repel:   float = None,
    ) -> Tuple[torch.Tensor, Dict]:
        B = z_f.size(0)
        w_e = w_erase if w_erase is not None else self.cfg.w_erase
        w_r = w_repel if w_repel is not None else self.cfg.w_repel

        # ── Anti-prototype push ────────────────────────────────────────────
        L_erase = (z_f * old_proto.unsqueeze(0)).sum(dim=1).mean()

        # ── KL-to-Uniform (calibration) ───────────────────────────────────
        p        = F.softmax(logits_f, dim=1)
        H        = -(p * torch.log(p + 1e-8)).sum(dim=1).mean()
        L_kl_uni = self.log_C - H

        # ── Pair repulsion with aggressive margin ─────────────────────────
        # margin = -0.3: penalise cos_sim > -0.3 (force strong scatter)
        if B > 1:
            sim_mat = z_f @ z_f.T                  # [B, B]
            mask    = ~torch.eye(B, dtype=torch.bool, device=z_f.device)
            L_repel = F.relu(sim_mat[mask] - self.cfg.repel_margin).mean()
        else:
            L_repel = torch.zeros(1, device=z_f.device).squeeze()

        total = (w_e                     * L_erase
               + self.cfg.w_kl_uniform   * L_kl_uni
               + w_r                     * L_repel)

        parts = {
            "L_erase":      float(L_erase.detach()),
            "L_kl_uni":     float(L_kl_uni.detach()),
            "L_repel":      float(L_repel.detach()),
            "entropy":      float(H.detach()),
            "cos_to_proto": float(L_erase.detach()),
        }
        return total, parts


class RetainLossV7(nn.Module):
    """
    Retain loss (v6 KD+anchor+CE) + retain-prototype anchor (mới trong v7.1).

    L_proto_retain là đối xứng trực tiếp với L_erase của ForgetLossV7:
      L_erase        = mean  cos(z_f, p_forget_old)   → ĐẨY forget ra xa prototype
      L_proto_retain = mean(1 - cos(z_r, p_{y_r}))    → KÉO retain VỀ prototype của
                                                          đúng lớp của nó (đóng băng
                                                          trước khi unlearn)
    Vẫn cùng một ý tưởng "local prototype" cốt lõi của LPEU — chỉ áp dụng theo
    hai chiều ngược nhau cho forget vs retain, thay vì chỉ có phía forget.
    Đây là cơ chế bảo vệ retain_accuracy trực tiếp nhất: ArcFace phân loại dựa
    trên cosine similarity với vector lớp, và việc neo embedding về đúng
    prototype của lớp giúp giữ nguyên vùng quyết định đó trong suốt quá trình
    neck bị chỉnh sửa bởi gradient forget.
    """
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.T   = cfg.kd_temperature
        self.ce  = nn.CrossEntropyLoss()

    def forward(
        self,
        logits_r:        torch.Tensor,
        logits_r_old:    torch.Tensor,
        y_r:             torch.Tensor,
        z_n_new:         Optional[torch.Tensor] = None,
        z_n_old:         Optional[torch.Tensor] = None,
        w_local:         float = None,   # override for staged schedule
        z_r_new:         Optional[torch.Tensor] = None,   # [B, D] retain embeddings (mới)
        retain_prototypes: Optional[torch.Tensor] = None, # [C, D] frozen, trên device
        w_proto:         float = None,   # override for staged schedule
    ) -> Tuple[torch.Tensor, Dict]:
        w_l = w_local if w_local is not None else self.cfg.w_kd_local
        w_p = w_proto if w_proto is not None else self.cfg.w_proto_retain

        # Global KL distillation
        log_p = F.log_softmax(logits_r     / self.T, dim=1)
        p_old = F.softmax(   logits_r_old  / self.T, dim=1)
        kl    = (self.T ** 2) * F.kl_div(log_p, p_old, reduction='batchmean')

        # CE
        ce = self.ce(logits_r, y_r)

        # Local K-NN cosine anchor (gated by w_l — zero in Phase 1)
        if z_n_new is not None and z_n_old is not None and w_l > 0:
            cos_sim  = (z_n_new * z_n_old).sum(dim=1)
            L_anchor = (1.0 - cos_sim).mean()
        else:
            L_anchor = torch.zeros(1, device=logits_r.device).squeeze()

        # Retain-prototype anchor (mới — luôn bật, kể cả Phase 1)
        if (self.cfg.use_proto_retain and z_r_new is not None
                and retain_prototypes is not None and w_p > 0):
            proto_y  = retain_prototypes[y_r]            # [B, D]
            cos_p    = (z_r_new * proto_y).sum(dim=1)
            L_proto  = (1.0 - cos_p).mean()
        else:
            L_proto = torch.zeros(1, device=logits_r.device).squeeze()

        total = (self.cfg.w_kd_global * kl
               + w_l                  * L_anchor
               + self.cfg.w_ce_retain * ce
               + w_p                  * L_proto)

        parts = {
            "L_kd":     float(kl.detach()),
            "L_anchor": float(L_anchor.detach()) if isinstance(L_anchor, torch.Tensor) else 0.0,
            "L_ce":     float(ce.detach()),
            "L_proto":  float(L_proto.detach()) if isinstance(L_proto, torch.Tensor) else 0.0,
        }
        return total, parts

In [15]:
# ===========================================================================
# ▓  CELL H2: PCGrad UTILITY
# ===========================================================================

def pcgrad_project(g_f: torch.Tensor, g_r: torch.Tensor) -> torch.Tensor:
    """
    Project g_f onto orthogonal complement of g_r when conflicting.
    g_f' = g_f − (⟨g_f,g_r⟩/‖g_r‖²)·g_r  if ⟨g_f,g_r⟩ < 0, else g_f.
    """
    inner = torch.dot(g_f, g_r)
    if inner >= 0:
        return g_f
    g_r_sq = torch.dot(g_r, g_r)
    if g_r_sq < 1e-12:
        return g_f
    return g_f - (inner / g_r_sq) * g_r

In [16]:
# ===========================================================================
# ▓  CELL I: STAGED UNLEARNING LOOP — LPEU-v7
# ===========================================================================

def get_stage_weights(epoch: int, cfg: Config) -> Tuple[float, float, float, float]:
    """
    Return (w_kd_local, w_erase, w_repel, pcgrad_boost) for the current epoch.

    Phase 1 — ERASE (epoch < stage1_epochs):
      Anchor OFF. Max forgetting. EWC acts as the only retain guard.
      Result: breaks the tight cluster without geometric deadlock.

    Phase 2 — REFINE (stage1 ≤ epoch < stage1+stage2):
      Anchor at 50%. Moderate forgetting. Cluster is already scattered;
      now we stabilize nearby identities while continuing to push forget cluster.

    Phase 3 — STABLE (epoch ≥ stage1+stage2):
      Anchor at 100%. Light forgetting. Nail down neighbor_shift < 0.15.
    """
    e1 = cfg.stage1_epochs
    e2 = cfg.stage1_epochs + cfg.stage2_epochs

    if epoch < e1:
        # Phase 1: ERASE
        return 0.0, cfg.w_erase, cfg.w_repel, cfg.pcgrad_boost
    elif epoch < e2:
        # Phase 2: REFINE
        return cfg.w_kd_local * 0.5, cfg.w_erase * 0.75, cfg.w_repel * 0.67, cfg.pcgrad_boost
    else:
        # Phase 3: STABLE
        return cfg.w_kd_local, cfg.w_erase * 0.5, cfg.w_repel * 0.33, cfg.pcgrad_boost * 0.5


def run_lpeu_v7_unlearning(
    model:              nn.Module,
    old_model:          nn.Module,
    forget_loader:      DataLoader,
    retain_loader:      DataLoader,
    retain_test_loader: DataLoader,
    anchor_loader:      Optional[DataLoader],
    forget_ids:         List[int],
    num_classes:        int,
    cfg:                Config,
    retain_acc_ref:     float,
    fisher:             Optional[Dict[str, torch.Tensor]] = None,
    theta_star:         Optional[Dict[str, torch.Tensor]] = None,
    retain_prototypes:  Optional[torch.Tensor] = None,   # [C, D] frozen, CPU
) -> nn.Module:
    """
    LPEU-v7 staged unlearning loop.

    Per-iteration:
    ═══════════════════════════════════════════════════════════════
    [A] RETAIN STEP:
      1. Forward retain batch → logits_r
      2. Forward anchor batch → z_n_new (Phase 2+3 only)
      3. L_retain = KL + staged_anchor + CE  [+ EWC if use_ewc]
      4. backward → save neck grads for PCGrad → step → STASH ArcFace

    [B] FORGET STEP (PCGrad protected):
      1. Forward forget batch → (z_f, logits_f)
      2. L_forget = staged_erase + kl_uni + staged_repel
      3. backward → PCGrad project neck grads → boost → step
      4. RESTORE ArcFace retain weights
    ═══════════════════════════════════════════════════════════════
    """
    device = cfg.device
    model.to(device)
    old_model.to(device)
    old_model.eval()
    for p in old_model.parameters():
        p.requires_grad = False

    # ── Trainable parameters ──────────────────────────────────────────────
    for p in model.parameters():
        p.requires_grad = False

    for name, p in model.named_parameters():
        if "neck" in name or "arcface" in name:
            p.requires_grad = True

    layer4_params = []
    for name, p in model.named_parameters():
        if "backbone.layer4" in name:
            p.requires_grad = True
            layer4_params.append(p)

    neck_arcface_params = [p for n, p in model.named_parameters()
                           if p.requires_grad and "backbone.layer4" not in n]
    all_trainable = [p for p in model.parameters() if p.requires_grad]

    pcgrad_param_names = set(n for n, p in model.named_parameters()
                             if p.requires_grad and ("neck" in n or "backbone.layer4" in n))

    n_train = sum(p.numel() for p in all_trainable)
    print(f"[v7] Trainable: {n_train:,} params  |  "
          f"EWC: {'ON' if cfg.use_ewc and fisher else 'OFF'}")

    # ── Optimizer ─────────────────────────────────────────────────────────
    optimizer = torch.optim.AdamW(
        [
            {"params": neck_arcface_params, "lr": cfg.unlearn_lr},
            {"params": layer4_params,       "lr": cfg.unlearn_lr * cfg.layer4_lr_mult},
        ],
        weight_decay=cfg.unlearn_wd,
        betas=(0.9, 0.999),
    )
    total_epochs = cfg.unlearn_epochs
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=total_epochs, eta_min=cfg.unlearn_lr * 0.05,
    )

    forget_crit = ForgetLossV7(cfg, num_classes)
    retain_crit = RetainLossV7(cfg)

    forget_proto = compute_forget_prototype(old_model, forget_loader, device)
    forget_proto = forget_proto.to(device)
    _p_cpu       = forget_proto.cpu()
    initial_proto_sim = float(
        compute_forget_prototype(old_model, forget_loader, device).dot(_p_cpu)
    )

    # Retain-prototype anchor: frozen [C, D] prototypes tính TRƯỚC unlearning
    retain_proto_dev = (
        retain_prototypes.to(device)
        if (cfg.use_proto_retain and retain_prototypes is not None) else None
    )

    fset = set(forget_ids)
    retain_cls_t = torch.tensor(
        [c for c in range(num_classes) if c not in fset], dtype=torch.long
    )

    best_state   = copy.deepcopy(model.state_dict())
    best_balance = -1.0
    violation_streak = 0

    print(f"\n[v7] Starting staged unlearning — {total_epochs} epochs total")
    print(f"     Phase 1 (ERASE):   epoch 0–{cfg.stage1_epochs-1}      anchor=OFF, repel={cfg.w_repel}")
    print(f"     Phase 2 (REFINE):  epoch {cfg.stage1_epochs}–{cfg.stage1_epochs+cfg.stage2_epochs-1}  anchor=50%, repel={cfg.w_repel*0.67:.1f}")
    print(f"     Phase 3 (STABLE):  epoch {cfg.stage1_epochs+cfg.stage2_epochs}–{total_epochs-1}  anchor=100%, repel={cfg.w_repel*0.33:.1f}")
    print(f"     Initial proto_sim = {initial_proto_sim:.4f}")

    anchor_iter = iter(anchor_loader) if anchor_loader else None

    for epoch in range(total_epochs):
        model.train()

        # ── Staged weight schedule ────────────────────────────────────────
        w_local, w_erase, w_repel, boost = get_stage_weights(epoch, cfg)
        phase_name = (
            "ERASE" if epoch < cfg.stage1_epochs else
            "REFINE" if epoch < cfg.stage1_epochs + cfg.stage2_epochs else
            "STABLE"
        )

        S = {k: 0. for k in ["kd","anchor","ce","proto","erase","kl_uni","repel","entropy","ewc","n_conf","steps"]}
        forget_iter = iter(forget_loader)

        for x_r, y_r, _ in retain_loader:
            x_r, y_r = x_r.to(device), y_r.to(device)

            # ══════════════════════════════════════════════════════════════
            # [A] RETAIN STEP
            # ══════════════════════════════════════════════════════════════
            z_r, logits_r = model(x_r, y_r)
            with torch.no_grad():
                _, logits_r_old = old_model(x_r, y_r)

            # Anchor batch (only in Phase 2+3 when w_local > 0)
            z_n_new = z_n_old = None
            if anchor_iter is not None and w_local > 0:
                try:
                    x_n, _, _ = next(anchor_iter)
                except StopIteration:
                    anchor_iter = iter(anchor_loader)
                    x_n, _, _ = next(anchor_iter)
                x_n = x_n.to(device)
                z_n_new, _ = model(x_n, None)
                with torch.no_grad():
                    z_n_old, _ = old_model(x_n, None)

            loss_r, parts_r = retain_crit(
                logits_r, logits_r_old, y_r, z_n_new, z_n_old,
                w_local=w_local,
                z_r_new=z_r, retain_prototypes=retain_proto_dev,
            )

            # Add EWC penalty (acts as data-free retain guard in Phase 1)
            ewc_val = 0.0
            if cfg.use_ewc and fisher is not None and theta_star is not None:
                L_ewc  = compute_ewc_loss(model, theta_star, fisher, cfg.w_ewc, device)
                loss_r = loss_r + L_ewc
                ewc_val = float(L_ewc.detach())

            optimizer.zero_grad()
            loss_r.backward()
            torch.nn.utils.clip_grad_norm_(all_trainable, cfg.grad_clip_retain)

            # Save retain grads for PCGrad
            retain_grads: Dict[str, torch.Tensor] = {}
            for name, param in model.named_parameters():
                if name in pcgrad_param_names and param.grad is not None:
                    retain_grads[name] = param.grad.detach().clone()

            optimizer.step()

            # Stash retain ArcFace weights
            stashed = model.arcface.weight.data[retain_cls_t].clone().cpu()

            S["kd"]     += parts_r["L_kd"]
            S["anchor"] += parts_r["L_anchor"]
            S["ce"]     += parts_r["L_ce"]
            S["proto"]  += parts_r["L_proto"]
            S["ewc"]    += ewc_val

            # ══════════════════════════════════════════════════════════════
            # [B] FORGET STEP — PCGrad protected
            # ══════════════════════════════════════════════════════════════
            try:
                x_f, _, _ = next(forget_iter)
            except StopIteration:
                forget_iter = iter(forget_loader)
                x_f, _, _ = next(forget_iter)
            x_f = x_f.to(device)

            z_f, logits_f = model(x_f, None)
            loss_f, parts_f = forget_crit(
                z_f, forget_proto, logits_f,
                w_erase=w_erase,
                w_repel=w_repel,
            )

            optimizer.zero_grad()
            loss_f.backward()
            torch.nn.utils.clip_grad_norm_(all_trainable, cfg.grad_clip_forget)

            # ── PCGrad surgery on neck + layer4 ────────────────────────────
            # v7.2 FIX: log thực tế cho thấy proto_sim gần như KHÔNG đổi suốt
            # 20 epoch dù boost=40× (xem comment ở forget_manual_lr trong
            # Config). Lý do: AdamW chuẩn hoá gradient theo m/√v nên nhân
            # gradient với hằng số boost gần như vô tác dụng khi đi qua
            # optimizer.step(). Ở đây ta TÍNH gradient đã PCGrad-chiếu + boost
            # như cũ, nhưng KHÔNG cho AdamW cập nhật 2 nhóm này (xoá .grad
            # trước khi step) — thay vào đó áp dụng SGD thủ công trực tiếp,
            # để boost thực sự tịnh tiến tham số tỉ lệ thuận như thiết kế.
            # ArcFace (không nằm trong pcgrad_param_names) vẫn update qua
            # AdamW như cũ — không ảnh hưởng.
            n_conflicts = 0
            manual_grads: Dict[str, torch.Tensor] = {}
            for name, param in model.named_parameters():
                if name not in pcgrad_param_names or param.grad is None:
                    continue
                g_f_flat = param.grad.detach().flatten()
                if name in retain_grads:
                    g_r_flat = retain_grads[name].flatten()
                    if torch.dot(g_f_flat, g_r_flat) < 0:
                        n_conflicts += 1
                    g_f_proj = pcgrad_project(g_f_flat, g_r_flat)
                else:
                    g_f_proj = g_f_flat
                manual_grads[name] = (g_f_proj * boost).reshape(param.shape)
                param.grad = None   # chặn AdamW cập nhật tham số này qua step()

            optimizer.step()   # chỉ còn tác dụng lên ArcFace (và các param khác nếu có)

            with torch.no_grad():
                for name, param in model.named_parameters():
                    if name in manual_grads:
                        lr_mult = cfg.layer4_lr_mult if "backbone.layer4" in name else 1.0
                        param -= cfg.forget_manual_lr * lr_mult * manual_grads[name]

            # Restore retain ArcFace weights
            model.arcface.weight.data[retain_cls_t] = stashed.to(device)

            S["erase"]   += parts_f["L_erase"]
            S["kl_uni"]  += parts_f["L_kl_uni"]
            S["repel"]   += parts_f["L_repel"]
            S["entropy"] += parts_f["entropy"]
            S["n_conf"]  += n_conflicts
            S["steps"]   += 1

            # ══════════════════════════════════════════════════════════════
            # [C] IN-LOOP RETAIN REPAIR (mới trong v7)
            # ══════════════════════════════════════════════════════════════
            # Sau mỗi forget step, chạy ngay một mini retain KD step để
            # "vá" damage từ forget gradient trước khi chuyển batch tiếp.
            # Dùng lại batch x_r đã có trên device — không tốn thêm data load.
            # Không dùng PCGrad ở bước này — pure retain recovery.
            if cfg.use_inloop_repair:
                _, logits_r_rep = model(x_r, y_r)
                with torch.no_grad():
                    _, logits_r_old_rep = old_model(x_r, y_r)
                log_p_rep = F.log_softmax(logits_r_rep / cfg.kd_temperature, dim=1)
                p_old_rep = F.softmax(logits_r_old_rep / cfg.kd_temperature, dim=1)
                loss_rep  = cfg.w_repair * (cfg.kd_temperature ** 2) * \
                            F.kl_div(log_p_rep, p_old_rep, reduction='batchmean')
                optimizer.zero_grad()
                loss_rep.backward()
                torch.nn.utils.clip_grad_norm_(all_trainable, cfg.grad_clip_retain)
                optimizer.step()
                # Stash-restore lại sau repair để arcface retain weights không drift
                model.arcface.weight.data[retain_cls_t] = stashed.to(device)

        scheduler.step()

        # ── Per-epoch eval ────────────────────────────────────────────────
        n          = max(1, S["steps"])
        ret_acc    = evaluate_accuracy(model, retain_test_loader, device)
        fgt_acc    = evaluate_accuracy(model, forget_loader, device)
        cur_proto  = compute_forget_prototype(model, forget_loader, device)  # CPU
        proto_sim  = float(cur_proto.dot(_p_cpu))
        proto_drop = initial_proto_sim - proto_sim

        print(
            f"[v7][{epoch+1:02d}/{total_epochs}][{phase_name}]  "
            f"erase={S['erase']/n:.4f}  repel={S['repel']/n:.4f}  "
            f"anc={S['anchor']/n:.4f}  proto_r={S['proto']/n:.4f}  ewc={S['ewc']/n:.4f}  "
            f"proto_sim={proto_sim:.4f}(drop:{proto_drop:+.4f})  "
            f"H={S['entropy']/n:.2f}/{math.log(num_classes):.2f}  "
            f"fgt_acc={fgt_acc:.4f}  ret_acc={ret_acc:.4f}  "
            f"conf={S['n_conf']/n:.1f}  boost={boost:.0f}×  "
            f"anchor_w={w_local:.1f}"
        )

        # ── Safety guardrail (v7.1: grace period + consecutive-violation patience) ──
        # Log thực tế của v7 gốc cho thấy ret_acc có thể chạm floor ngay epoch 2
        # (stage1_epochs=1) và revert model về gần như CHƯA unlearn gì cả — vô
        # hiệu hoá toàn bộ cơ chế clustering. v7.1 cho một khoảng grace + yêu cầu
        # vi phạm liên tiếp trước khi thực sự dừng, để in-loop repair / EWC /
        # retain-prototype anchor có cơ hội kéo ret_acc hồi phục trước khi bỏ cuộc.
        floor = cfg.retain_acc_floor * retain_acc_ref
        if epoch >= cfg.safety_grace_epochs and ret_acc < floor:
            violation_streak += 1
            print(f"[v7] ⚠ retain_acc dưới floor ({ret_acc:.4f} < {floor:.4f})  "
                  f"streak={violation_streak}/{cfg.safety_patience}")
            if violation_streak >= cfg.safety_patience:
                print(f"[v7] ⚠ SAFETY STOP epoch {epoch+1}: vi phạm floor "
                      f"{cfg.safety_patience} epoch liên tiếp")
                model.load_state_dict(best_state)
                break
        else:
            violation_streak = 0

        # Best checkpoint: maximize forget_score × retain_ratio
        retain_score = ret_acc / retain_acc_ref
        forget_score = max(0.0, proto_drop / (1.0 + initial_proto_sim))
        balance = forget_score * retain_score

        if balance > best_balance:
            best_balance = balance
            best_state   = copy.deepcopy(model.state_dict())
            print(f"          ★ New best  balance={best_balance:.4f}  "
                  f"proto_drop={proto_drop:+.4f}  forget_score={forget_score:.3f}  "
                  f"retain={retain_score:.3f}")

    model.load_state_dict(best_state)
    for p in model.parameters():
        p.requires_grad = True
    print(f"\n[v7] Done. Best balance: {best_balance:.4f}")
    return model

In [17]:
# ===========================================================================
# ▓  CELL I2: POST-UNLEARNING RETAIN REPAIR
# ===========================================================================

def run_retain_repair(
    model:              nn.Module,
    old_model:          nn.Module,
    retain_loader:      DataLoader,
    retain_test_loader: DataLoader,
    forget_ids:         List[int],
    num_classes:        int,
    cfg:                Config,
    retain_prototypes:  Optional[torch.Tensor] = None,   # [C, D] frozen, CPU
    retain_val_loader:  Optional[DataLoader]    = None,   # v7.3: best-ckpt selection
) -> nn.Module:
    """
    Two-phase post-unlearning retain repair.

    ROOT CAUSE FIX vs previous version:
    ─────────────────────────────────────
    Old version restored retain arcface weights → original after every step.
    Problem: unlearning shifts the neck to a new position. Using original arcface
    weights (calibrated for the old neck) with the new neck creates a mismatch →
    retain_acc can never exceed original.

    New approach:
    ─────────────────────────────────────────────────────────────────────────────
    Phase R1 (60% epochs, high LR):
      - ONLY freeze FORGET arcface weights (prevent re-learning forget IDs)
      - Let neck + RETAIN arcface weights adapt together
      - CE dominant — directly optimize classification accuracy
      - Cosine LR decay for smooth convergence

    Phase R2 (40% epochs, low LR):
      - KD stabilization — prevent over-drift from original representations
      - CE continues to maintain boundaries
      - Still only freeze forget arcface

    Why this is safe (forget is NOT re-learned):
    1. Forget samples absent from retain_loader → no gradient signal for forget
    2. Forget arcface weights frozen throughout → forget class boundaries unchanged
    3. Neck adapts only via retain gradients → forget embeddings drift with neck
       but their arcface weights are stale → forget classification stays broken

    v7.3 FIX — best-checkpoint selection (giống train_original_model):
    ─────────────────────────────────────────────────────────────────────────────
    Log Korean Family thực tế cho thấy retain_acc DAO ĐỘNG suốt R1/R2 (có epoch
    đạt 0.1343 giữa chừng) nhưng hàm CŨ chỉ trả về state của EPOCH CUỐI CÙNG —
    dù epoch cuối tệ hơn 1 epoch giữa chừng (0.1110 < 0.1343 đã từng đạt). Đây
    là nguyên nhân trực tiếp khiến retain_accuracy cuối cùng thấp hơn mức model
    ĐÃ TỪNG đạt được trong chính quá trình repair. Fix: theo dõi best checkpoint
    trên retain_val_loader (KHÔNG dùng retain_test_loader để chọn — tránh leak
    test set vào lựa chọn checkpoint, giữ so sánh công bằng với FineTune/NegGrad
    vốn không có bước chọn checkpoint nào), rồi load lại best_state trước khi
    trả về model. Đây là thay đổi CHỈ CÓ THỂ GIÚP, không thể làm hại — trường
    hợp xấu nhất, epoch cuối vốn đã là tốt nhất thì kết quả không đổi.
    """
    device = cfg.device
    best_val_acc, best_state = -1.0, None
    model.to(device)
    old_model.eval()
    for p in old_model.parameters():
        p.requires_grad = False

    fset = set(forget_ids)
    forget_cls_t = torch.tensor(list(fset), dtype=torch.long)

    # Stash forget arcface weights (state after unlearning — NOT original)
    stashed_forget_arcface = model.arcface.weight.data[forget_cls_t].clone()

    # v7.1: retain-prototype anchor cũng áp dụng trong bước repair — cùng ý
    # tưởng "neo về prototype đúng lớp" như trong RetainLossV7, giúp CE hội tụ
    # nhanh hơn về đúng vùng quyết định thay vì chỉ dựa vào KD+CE trần.
    proto_dev = (
        retain_prototypes.to(device)
        if (cfg.use_proto_retain and retain_prototypes is not None) else None
    )

    T  = cfg.kd_temperature
    ce = nn.CrossEntropyLoss()

    r1_epochs = max(1, int(cfg.repair_epochs * 0.6))
    r2_epochs = cfg.repair_epochs - r1_epochs

    print(f"\n[Repair] Two-phase retain repair — {cfg.repair_epochs} epochs total")
    print(f"         R1={r1_epochs}e (LR={cfg.repair_lr:.1e}, CE+KD, neck+retain-arcface adapt, "
          f"layer4 LR={cfg.repair_lr*cfg.repair_layer4_lr_mult:.1e})")
    print(f"         R2={r2_epochs}e (LR={cfg.repair_lr*0.1:.1e}, KD stabilisation)")
    print(f"         KEY: only FORGET arcface frozen; retain arcface + layer4 free to adapt")

    # ── PHASE R1: Adaptive neck + retain-arcface + layer4 re-alignment ──────
    # v7.3: layer4 được thêm vào (LR nhỏ hơn) để "vá" đúng phần mà erase step
    # đã chỉnh — xem comment ở repair_layer4_lr_mult trong Config.
    trainable_r1 = [p for n, p in model.named_parameters()
                    if ("neck" in n or "arcface" in n) and "backbone.layer4" not in n]
    layer4_r1    = [p for n, p in model.named_parameters() if "backbone.layer4" in n]
    all_r1_params = trainable_r1 + layer4_r1
    opt_r1 = torch.optim.AdamW(
        [
            {"params": trainable_r1, "lr": cfg.repair_lr},
            {"params": layer4_r1,    "lr": cfg.repair_lr * cfg.repair_layer4_lr_mult},
        ],
        weight_decay=1e-5
    )
    sched_r1 = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt_r1, T_max=max(1, r1_epochs), eta_min=cfg.repair_lr * 0.1
    )

    for epoch in range(r1_epochs):
        model.train()
        total_loss = 0.0
        for x_r, y_r, _ in retain_loader:
            x_r, y_r = x_r.to(device), y_r.to(device)

            z_r, logits_new = model(x_r, y_r)

            # CE: direct accuracy objective — dominant in R1
            loss_ce = cfg.w_ce_retain * ce(logits_new, y_r)

            # Light KD: soft guard against catastrophic over-fitting
            with torch.no_grad():
                _, logits_old = old_model(x_r, y_r)
            log_p = F.log_softmax(logits_new / T, dim=1)
            p_old = F.softmax(logits_old   / T, dim=1)
            loss_kl = 5.0 * (T ** 2) * F.kl_div(log_p, p_old, reduction='batchmean')

            # Retain-prototype anchor — kéo embedding về đúng prototype gốc
            if proto_dev is not None:
                proto_y = proto_dev[y_r]
                loss_proto = cfg.w_proto_retain * (1.0 - (z_r * proto_y).sum(dim=1)).mean()
            else:
                loss_proto = torch.zeros(1, device=device).squeeze()

            loss = loss_ce + loss_kl + loss_proto

            opt_r1.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(all_r1_params, 1.0)
            opt_r1.step()

            # Only restore FORGET arcface (retain arcface adapts freely)
            model.arcface.weight.data[forget_cls_t] = stashed_forget_arcface.to(device)

            total_loss += float(loss)

        sched_r1.step()
        val_source = retain_val_loader if retain_val_loader is not None else retain_test_loader
        ret_acc = evaluate_accuracy(model, val_source, device)
        if ret_acc > best_val_acc:
            best_val_acc = ret_acc
            best_state = copy.deepcopy(model.state_dict())
        print(f"[R1][{epoch+1:02d}/{r1_epochs}]  "
              f"loss={total_loss/len(retain_loader):.4f}  "
              f"retain_val_acc={ret_acc:.4f}  best={best_val_acc:.4f}")

    # ── PHASE R2: KD Stabilisation ────────────────────────────────────────────
    trainable_r2 = [p for n, p in model.named_parameters()
                    if ("neck" in n or "arcface" in n) and "backbone.layer4" not in n]
    layer4_r2    = [p for n, p in model.named_parameters() if "backbone.layer4" in n]
    all_r2_params = trainable_r2 + layer4_r2
    opt_r2 = torch.optim.AdamW(
        [
            {"params": trainable_r2, "lr": cfg.repair_lr * 0.1},
            {"params": layer4_r2,    "lr": cfg.repair_lr * 0.1 * cfg.repair_layer4_lr_mult},
        ],
        weight_decay=1e-5
    )

    for epoch in range(r2_epochs):
        model.train()
        total_loss = 0.0
        for x_r, y_r, _ in retain_loader:
            x_r, y_r = x_r.to(device), y_r.to(device)

            z_r, logits_new = model(x_r, y_r)
            with torch.no_grad():
                _, logits_old = old_model(x_r, y_r)

            # KD dominant in R2 — stabilise without over-drifting.
            # repair_r2_kd_mult cho phép hạ lực KD riêng cho dataset khó
            # (xem comment ở Config) mà không ảnh hưởng dataset đang chạy tốt.
            log_p = F.log_softmax(logits_new / T, dim=1)
            p_old = F.softmax(logits_old   / T, dim=1)
            loss_kl = (cfg.w_kd_global * cfg.repair_r2_kd_mult
                       * (T ** 2) * F.kl_div(log_p, p_old, reduction='batchmean'))

            loss_ce = cfg.w_ce_retain * ce(logits_new, y_r)

            if proto_dev is not None:
                proto_y = proto_dev[y_r]
                loss_proto = (cfg.w_proto_retain * 0.5) * (1.0 - (z_r * proto_y).sum(dim=1)).mean()
            else:
                loss_proto = torch.zeros(1, device=device).squeeze()

            loss = loss_kl + loss_ce + loss_proto

            opt_r2.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(all_r2_params, cfg.grad_clip_retain)
            opt_r2.step()

            # Keep forget arcface frozen
            model.arcface.weight.data[forget_cls_t] = stashed_forget_arcface.to(device)

            total_loss += float(loss)

        val_source = retain_val_loader if retain_val_loader is not None else retain_test_loader
        ret_acc = evaluate_accuracy(model, val_source, device)
        if ret_acc > best_val_acc:
            best_val_acc = ret_acc
            best_state = copy.deepcopy(model.state_dict())
        print(f"[R2][{epoch+1:02d}/{r2_epochs}]  "
              f"loss={total_loss/len(retain_loader):.4f}  "
              f"retain_val_acc={ret_acc:.4f}  best={best_val_acc:.4f}")

    # v7.3: khôi phục checkpoint TỐT NHẤT theo retain_val_loader, không phải
    # state của epoch cuối cùng — xem lý do ở docstring đầu hàm.
    if best_state is not None:
        model.load_state_dict(best_state)
    final_acc = evaluate_accuracy(model, retain_test_loader, device)
    print(f"\n[Repair] Done. Best retain_val_acc={best_val_acc:.4f} → "
          f"Final retain_TEST_acc={final_acc:.4f}")
    return model

In [18]:
# ===========================================================================
# ▓  CELL J: EVALUATION METRICS
# ===========================================================================

@torch.no_grad()
def evaluate_accuracy(model, loader, device):
    model.eval(); correct = total = 0
    for x, y, _ in loader:
        x, y = x.to(device), y.to(device)
        _, logits = model(x, None)
        correct += int((logits.argmax(1) == y).sum())
        total   += y.numel()
    return correct / max(1, total)

@torch.no_grad()
def mean_sim_to_proto(model, loader, proto, device):
    model.eval(); sims = []
    for x, _, _ in loader:
        z, _ = model(x.to(device), None)
        sims.append((z @ proto.to(device)).mean().item())
    return float(np.mean(sims)) if sims else 0.0

@torch.no_grad()
def mean_entropy(model, loader, device):
    model.eval(); vals = []
    for x, _, _ in loader:
        _, logits = model(x.to(device), None)
        p = F.softmax(logits, dim=1)
        vals.append(-(p * torch.log(p + 1e-8)).sum(1).mean().item())
    return float(np.mean(vals)) if vals else 0.0

@torch.no_grad()
def cluster_compactness(model, loader, device):
    model.eval()
    embs = [model(x.to(device), None)[0].cpu() for x, _, _ in loader]
    z = torch.cat(embs)
    return float(((z - z.mean(0, keepdim=True))**2).sum(1).mean())

@torch.no_grad()
def intra_cluster_cos_sim(model, loader, device):
    model.eval()
    embs = [model(x.to(device), None)[0].cpu() for x, _, _ in loader]
    z = F.normalize(torch.cat(embs), dim=1)
    if z.size(0) < 2: return 1.0
    sim_mat = z @ z.T
    mask = ~torch.eye(z.size(0), dtype=torch.bool)
    return float(sim_mat[mask].mean())

@torch.no_grad()
def neighbor_shift(new_m, old_m, loader, device):
    new_m.eval(); old_m.eval(); shifts = []
    for x, _, _ in loader:
        x = x.to(device)
        z_new, _ = new_m(x, None); z_old, _ = old_m(x, None)
        shifts.append((z_new - z_old).norm(dim=1).mean().item())
    return float(np.mean(shifts)) if shifts else 0.0

@torch.no_grad()
def get_loss_signal(model, loader, device, n_max=500):
    model.eval(); scores = []
    ce = nn.CrossEntropyLoss(reduction='none')
    with torch.no_grad():
        for x, y, _ in loader:
            x, y = x.to(device), y.to(device)
            _, logits = model(x, None)
            loss = ce(logits, y)
            scores.append((-loss).cpu())
            if sum(len(s) for s in scores) >= n_max: break
    return torch.cat(scores)[:n_max]

def compute_mia_auc(model, forget_loader, retain_test_loader, device, cfg):
    n   = min(cfg.mia_n_members, cfg.mia_n_nonmembers)
    mem = get_loss_signal(model, forget_loader,      device, n)
    non = get_loss_signal(model, retain_test_loader, device, n)
    X   = torch.cat([mem, non]).unsqueeze(1).numpy()
    y   = np.array([1]*len(mem) + [0]*len(non))
    if len(np.unique(y)) < 2: return 0.5
    try:
        clf = LogisticRegression(max_iter=500)
        clf.fit(X, y)
        return float(roc_auc_score(y, clf.predict_proba(X)[:,1]))
    except: return 0.5


def full_evaluate(
    name: str,
    model: nn.Module,
    old_model: nn.Module,
    retain_test_loader: DataLoader,
    forget_eval_loader: DataLoader,
    local_loader: DataLoader,
    old_proto: torch.Tensor,
    cfg: Config,
    num_classes: int,
) -> Dict:
    device = cfg.device
    rc     = 1.0 / num_classes

    ret_acc  = evaluate_accuracy(model, retain_test_loader, device)
    fgt_acc  = evaluate_accuracy(model, forget_eval_loader, device)
    old_sim  = mean_sim_to_proto(old_model, forget_eval_loader, old_proto, device)
    new_sim  = mean_sim_to_proto(model,     forget_eval_loader, old_proto, device)
    nshift   = neighbor_shift(model, old_model, local_loader, device)
    old_ent  = mean_entropy(old_model, forget_eval_loader, device)
    new_ent  = mean_entropy(model,     forget_eval_loader, device)
    old_comp = cluster_compactness(old_model, forget_eval_loader, device)
    new_comp = cluster_compactness(model,     forget_eval_loader, device)
    old_ics  = intra_cluster_cos_sim(old_model, forget_eval_loader, device)
    new_ics  = intra_cluster_cos_sim(model,     forget_eval_loader, device)
    mia      = compute_mia_auc(model, forget_eval_loader, retain_test_loader, device, cfg)

    m = dict(
        retain_accuracy       = ret_acc,
        forget_accuracy       = fgt_acc,
        forget_acc_vs_random  = fgt_acc - rc,
        old_forget_sim        = old_sim,
        new_forget_sim        = new_sim,
        forget_sim_drop       = old_sim - new_sim,
        neighbor_shift        = nshift,
        old_entropy           = old_ent,
        new_entropy           = new_ent,
        entropy_increase      = new_ent - old_ent,
        old_compactness       = old_comp,
        new_compactness       = new_comp,
        compactness_increase  = new_comp - old_comp,
        old_intra_cluster_sim = old_ics,
        new_intra_cluster_sim = new_ics,
        cluster_sim_drop      = old_ics - new_ics,
        mia_auc               = mia,
    )

    bar = "═" * 70
    tgt_fgt = "✓" if abs(fgt_acc - rc) < 0.01 else ("⚠ low"  if fgt_acc < rc*0.5 else "⚠ high")
    tgt_sim = "✓" if (old_sim - new_sim) > 0.25  else "·"
    tgt_ns  = "✓" if nshift < 0.15               else "⚠"
    tgt_ent = "✓" if (new_ent - old_ent) > 0.5   else "·"
    tgt_mia = "✓" if abs(mia - 0.5) < 0.03       else "·"
    tgt_ics = "✓" if (old_ics - new_ics) > 0.20  else "·"

    print(f"\n{bar}")
    print(f"  [{name}]")
    print(f"{bar}")
    print(f"  Retain Accuracy      : {ret_acc:.4f}          ← HIGH is good")
    print(f"  Forget Accuracy      : {fgt_acc:.4f}  {tgt_fgt}  ← ~{rc:.4f} ideal (1/C)")
    print(f"  Forget Sim Drop  ↑   : {old_sim-new_sim:+.4f}  {tgt_sim}  ← ≥ +0.25 target")
    print(f"  Cluster Cos Drop ↑   : {old_ics-new_ics:+.4f}  {tgt_ics}  ← ≥ +0.20 target")
    print(f"  Compactness Incr ↑   : {new_comp-old_comp:+.4f}          ← positive = dispersed")
    print(f"  Entropy Increase ↑   : {new_ent-old_ent:+.4f}  {tgt_ent}  ← ≥ +0.50 target")
    print(f"  Neighbor Shift   ↓   : {nshift:.4f}  {tgt_ns}  ← < 0.15 target")
    print(f"  MIA AUC          →0.5: {mia:.4f}  {tgt_mia}  ← 0.5 = ideal privacy")
    print(f"{bar}")
    return m

In [19]:
# ===========================================================================
# ▓  CELL K: BASELINES
# ===========================================================================

def run_finetune(model, retain_loader, cfg, num_epochs=5):
    device = cfg.device; model.to(device)
    opt = torch.optim.SGD(model.parameters(), lr=1e-4, momentum=0.9, weight_decay=1e-4)
    ce  = nn.CrossEntropyLoss()
    for _ in range(num_epochs):
        model.train()
        for x, y, _ in retain_loader:
            x, y = x.to(device), y.to(device)
            _, logits = model(x, y)
            loss = ce(logits, y)
            opt.zero_grad(); loss.backward(); opt.step()
    return model


def run_neggrad(model, forget_loader, retain_loader, cfg, num_epochs=5):
    device = cfg.device; model.to(device)
    opt = torch.optim.SGD(model.parameters(), lr=1e-5, momentum=0.9)
    ce  = nn.CrossEntropyLoss()
    forget_iter = iter(forget_loader)
    for _ in range(num_epochs):
        model.train()
        for x_r, y_r, _ in retain_loader:
            x_r, y_r = x_r.to(device), y_r.to(device)
            _, logits_r = model(x_r, y_r); loss_r = ce(logits_r, y_r)
            try: x_f, y_f, _ = next(forget_iter)
            except StopIteration:
                forget_iter = iter(forget_loader)
                x_f, y_f, _ = next(forget_iter)
            x_f, y_f = x_f.to(device), y_f.to(device)
            _, logits_f = model(x_f, y_f)
            loss = loss_r - ce(logits_f, y_f)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
    return model

In [20]:
import os
for f in ["original_model.pt", "lpeu_v7_model.pt", "lpeu_v7_repaired_model.pt", "results_v7.json"]:
    p = os.path.join("./runs/lpeu_v7_vggface2", f)
    if os.path.exists(p): os.remove(p)

In [21]:
# ===========================================================================
# ▓  CELL L: FULL PIPELINE
# ===========================================================================

# ─── Step 1: Dataset ─────────────────────────────────────────────────────────
print("\n[Step 1] Loading dataset...")
train_base, eval_base, filtered_indices, valid_classes = load_full_dataset(cfg)
num_classes = len(valid_classes)
split = split_by_identity(train_base, filtered_indices, cfg)

print(f"  Identities: {num_classes}  |  Forget IDs: {split['forget_ids']}")
print(f"  Forget train: {len(split['forget_idx'])}  |  "
      f"Retain train: {len(split['retain_train_idx'])}  |  "
      f"Retain test: {len(split['retain_test_idx'])}")

# ─── Step 2: Loaders ─────────────────────────────────────────────────────────
train_ds          = IndexedSubset(train_base, split["train_idx"])
val_ds            = IndexedSubset(eval_base,  split["val_idx"])
retain_train_ds   = IndexedSubset(train_base, split["retain_train_idx"])
retain_test_ds    = IndexedSubset(eval_base,  split["retain_test_idx"])
retain_val_ds     = IndexedSubset(eval_base,  split["retain_val_idx"])
forget_train_ds   = IndexedSubset(train_base, split["forget_idx"])
forget_eval_ds    = IndexedSubset(eval_base,  split["forget_idx"])

train_loader        = make_loader(train_ds,       cfg.batch_size, True,  cfg.num_workers)
val_loader          = make_loader(val_ds,          cfg.batch_size, False, cfg.num_workers)
retain_train_loader = make_loader(retain_train_ds, cfg.batch_size, True,  cfg.num_workers)
retain_test_loader  = make_loader(retain_test_ds,  cfg.batch_size, False, cfg.num_workers)
retain_val_loader   = make_loader(retain_val_ds,   cfg.batch_size, False, cfg.num_workers)
forget_train_loader = make_loader(forget_train_ds, cfg.batch_size, True,  cfg.num_workers)
forget_eval_loader  = make_loader(forget_eval_ds,  cfg.batch_size, False, cfg.num_workers)

# ─── Step 3: Train or load original model ────────────────────────────────────
print("\n[Step 2] Training original model  [arcface_s=32, m=0.30, Simple Neck]...")
original_model = FaceModel(
    num_classes=num_classes,
    embedding_dim=cfg.embedding_dim,
    pretrained=cfg.pretrained_backbone,
    s=cfg.arcface_s, m=cfg.arcface_m,
)

orig_ckpt = os.path.join(cfg.output_dir, "original_model.pt")
if os.path.exists(orig_ckpt):
    print(f"  → Loading cached model from {orig_ckpt}")
    print(f"  ⚠ If this is a v6 model (BN-Neck), DELETE it:")
    print(f"     import os; os.remove('{orig_ckpt}')")
    original_model.load_state_dict(torch.load(orig_ckpt, map_location=cfg.device))
    original_model.to(cfg.device)
else:
    original_model = train_original_model(original_model, train_loader, val_loader, cfg)
    torch.save(original_model.state_dict(), orig_ckpt)
    print(f"  → Saved to {orig_ckpt}")

# ─── Step 4: Prototypes & K-NN anchor ────────────────────────────────────────
print("\n[Step 3] Computing prototypes & K-NN anchor...")
all_prototypes = compute_class_prototypes(
    original_model,
    make_loader(IndexedSubset(eval_base, split["train_idx"]),
                cfg.batch_size, False, cfg.num_workers),
    num_classes, cfg.device,
)
old_forget_proto = compute_forget_prototype(original_model, forget_eval_loader, cfg.device)

anchor_class_ids = find_knn_retain_classes(
    forget_proto=old_forget_proto,
    all_prototypes=all_prototypes,
    forget_ids=split["forget_ids"],
    k=cfg.k_anchor,
)
print(f"  Forget prototype ‖p‖ = {old_forget_proto.norm():.4f}")
print(f"  K-NN anchor classes  : {anchor_class_ids}")

# Verify intra_cluster_sim — key health check for the model
ics = intra_cluster_cos_sim(original_model, forget_eval_loader, cfg.device)
fsp = mean_sim_to_proto(original_model, forget_eval_loader, old_forget_proto, cfg.device)
print(f"\n  ── Model Health Check ──────────────────────────────────")
print(f"  old_forget_sim       = {fsp:.4f}   (target: 0.5–0.8)")
print(f"  intra_cluster_sim    = {ics:.4f}   (target: 0.30–0.60)")
if ics > 0.80:
    print(f"  ⚠ intra_cluster_sim TOO HIGH ({ics:.3f} > 0.80)")
    print(f"    This means embedding collapse. The model will be hard to unlearn.")
    print(f"    → Make sure you're using v7 FaceModel (Simple Neck, NOT BN-Neck)")
    print(f"    → Delete the cached model and retrain.")
else:
    print(f"  ✓ intra_cluster_sim looks healthy")
print(f"  ──────────────────────────────────────────────────────")

anchor_loader = build_local_anchor_loader(
    eval_base, split["retain_train_idx"],
    anchor_class_ids, cfg.batch_size, cfg.num_workers,
)

local_neighbor_loader = make_loader(
    IndexedSubset(eval_base, [
        i for i in split["retain_train_idx"]
        if eval_base.samples[i][1] in anchor_class_ids
    ]),
    cfg.batch_size, False, cfg.num_workers,
)

# ─── Step 5: EWC Fisher computation ──────────────────────────────────────────
print("\n[Step 4] Computing EWC Fisher Information...")
fisher = theta_star = None
if cfg.use_ewc:
    theta_star = snapshot_neck_params(original_model)
    fisher     = compute_ewc_fisher(
        original_model, retain_train_loader, cfg.device, n_batches=30
    )
else:
    print("  EWC disabled (use_ewc=False).")

# ─── Step 6: Baselines ───────────────────────────────────────────────────────
print("\n[Step 5] Running baselines...")
ft_model = run_finetune(clone_model(original_model), retain_train_loader, cfg)
ng_model = run_neggrad(clone_model(original_model), forget_train_loader, retain_train_loader, cfg)

# ─── Step 7: LPEU-v7 ─────────────────────────────────────────────────────────
print("\n[Step 6] Running LPEU-v7...")
retain_acc_ref = evaluate_accuracy(original_model, retain_test_loader, cfg.device)
print(f"  Reference retain accuracy: {retain_acc_ref:.4f}")

lpeu_v7_model = run_lpeu_v7_unlearning(
    model               = clone_model(original_model),
    old_model           = clone_model(original_model),
    forget_loader       = forget_train_loader,
    retain_loader       = retain_train_loader,
    retain_test_loader  = retain_test_loader,
    anchor_loader       = anchor_loader,
    forget_ids          = split["forget_ids"],
    num_classes         = num_classes,
    cfg                 = cfg,
    retain_acc_ref      = retain_acc_ref,
    fisher              = fisher,
    theta_star          = theta_star,
    retain_prototypes   = all_prototypes,
)
torch.save(lpeu_v7_model.state_dict(),
           os.path.join(cfg.output_dir, "lpeu_v7_model.pt"))
print("Saved LPEU-v7 model (before repair).")

# ─── Post-unlearning retain repair ───────────────────────────────────────────
if cfg.use_post_repair:
    print("\n[Step 6b] Post-unlearning retain repair...")
    lpeu_v7_model = run_retain_repair(
        model               = lpeu_v7_model,
        old_model           = clone_model(original_model),
        retain_loader       = retain_train_loader,
        retain_test_loader  = retain_test_loader,
        forget_ids          = split["forget_ids"],
        num_classes         = num_classes,
        cfg                 = cfg,
        retain_prototypes   = all_prototypes,
        retain_val_loader   = retain_val_loader,
    )
    torch.save(lpeu_v7_model.state_dict(),
               os.path.join(cfg.output_dir, "lpeu_v7_repaired_model.pt"))
    print("Saved LPEU-v7 repaired model.")

# ─── Step 8: Evaluation ──────────────────────────────────────────────────────
print("\n[Step 7] Full evaluation...")

eval_kwargs = dict(
    old_model          = original_model,
    retain_test_loader = retain_test_loader,
    forget_eval_loader = forget_eval_loader,
    local_loader       = local_neighbor_loader,
    old_proto          = old_forget_proto,
    cfg                = cfg,
    num_classes        = num_classes,
)

orig_m = full_evaluate("Original",  original_model, **eval_kwargs)
ft_m   = full_evaluate("FineTune",  ft_model,       **eval_kwargs)
ng_m   = full_evaluate("NegGrad",   ng_model,        **eval_kwargs)
v7_m   = full_evaluate("LPEU-v7 ★", lpeu_v7_model,  **eval_kwargs)

results = dict(
    original  = orig_m,
    finetune  = ft_m,
    neggrad   = ng_m,
    lpeu_v7   = v7_m,
    forget_ids = split["forget_ids"],
    config = dict(
        embedding_dim  = cfg.embedding_dim,
        arcface_s      = cfg.arcface_s,
        arcface_m      = cfg.arcface_m,
        k_anchor       = cfg.k_anchor,
        w_kd_global    = cfg.w_kd_global,
        w_kd_local     = cfg.w_kd_local,
        w_ewc          = cfg.w_ewc,
        w_erase        = cfg.w_erase,
        w_kl_uniform   = cfg.w_kl_uniform,
        w_repel        = cfg.w_repel,
        repel_margin   = cfg.repel_margin,
        pcgrad_boost   = cfg.pcgrad_boost,
        stage1_epochs  = cfg.stage1_epochs,
        stage2_epochs  = cfg.stage2_epochs,
        stage3_epochs  = cfg.stage3_epochs,
        w_proto_retain = cfg.w_proto_retain,
        original_epochs      = cfg.original_epochs,
        early_stop_patience  = cfg.early_stop_patience,
        safety_grace_epochs  = cfg.safety_grace_epochs,
        safety_patience      = cfg.safety_patience,
        forget_manual_lr     = cfg.forget_manual_lr,
    )
)
out_path = os.path.join(cfg.output_dir, "results_v7.json")
with open(out_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"\nSaved results → {out_path}")



[Step 1] Loading dataset...
  Identities: 300  |  Forget IDs: [41, 270, 251, 236, 136, 34, 230, 199, 98, 141, 73, 186, 30, 242, 126, 114, 1, 167, 182, 247, 287, 74, 139, 163, 248, 160, 274, 102, 96, 61]
  Forget train: 1260  |  Retain train: 11340  |  Retain test: 3240

[Step 2] Training original model  [arcface_s=32, m=0.30, Simple Neck]...
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 162MB/s]
/tmp/ipykernel_22/3990456301.py:71: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  total_loss += float(loss)


[Train][05/120]  loss=7.6149  val_acc=0.0506  best_val=0.1394  no_improve=4  lr=5.00e-02
[Train]   ↓ LR giảm còn 2.50e-02 (no_improve=8, best_val=0.1394)
[Train][10/120]  loss=7.4172  val_acc=0.1189  best_val=0.1394  no_improve=9  lr=2.50e-02
[Train]   ↓ LR giảm còn 1.25e-02 (no_improve=12, best_val=0.1394)
[Train][15/120]  loss=7.3181  val_acc=0.1511  best_val=0.1644  no_improve=1  lr=1.25e-02
[Train][20/120]  loss=7.2969  val_acc=0.1233  best_val=0.1722  no_improve=2  lr=1.25e-02
[Train]   ↓ LR giảm còn 6.25e-03 (no_improve=4, best_val=0.1722)
[Train][25/120]  loss=7.2574  val_acc=0.1872  best_val=0.1922  no_improve=2  lr=6.25e-03
[Train]   ↓ LR giảm còn 3.13e-03 (no_improve=4, best_val=0.1956)
[Train][30/120]  loss=7.2524  val_acc=0.1783  best_val=0.1956  no_improve=4  lr=3.13e-03
[Train]   ↓ LR giảm còn 1.56e-03 (no_improve=4, best_val=0.2250)
[Train][35/120]  loss=7.2356  val_acc=0.2228  best_val=0.2250  no_improve=4  lr=1.56e-03
[Train]   ↓ LR giảm còn 7.81e-04 (no_improve=4, bes

In [22]:
# ===========================================================================
# ▓  CELL M: ABLATION STUDY
# ===========================================================================

def run_ablation(name: str, cfg_overrides: dict) -> Dict:
    """
    Run LPEU-v7 with specific components disabled.

    Recommended ablations for paper Table:
    ─────────────────────────────────────
    A1: No staging (flat weights from epoch 0)
        run_ablation("NoStaging", {"stage1_epochs": 0, "stage2_epochs": 0,
                                   "stage3_epochs": 20})

    A2: No EWC
        run_ablation("NoEWC", {"use_ewc": False})

    A3: No PCGrad
        run_ablation("NoPCGrad", {"use_pcgrad": False, "pcgrad_boost": 1.0})

    A4: No K-NN anchor (all phases)
        run_ablation("NoAnchor", {"w_kd_local": 0.0})

    A5: Standard margin (0.0 instead of -0.3)
        run_ablation("MarginZero", {"repel_margin": 0.0})

    A6: No pair repulsion
        run_ablation("NoRepulsion", {"w_repel": 0.0})

    A7: No retain-prototype anchor (v7.1)
        run_ablation("NoProtoRetain", {"use_proto_retain": False})
    """
    cfg_abl = copy.deepcopy(cfg)
    for k, v in cfg_overrides.items():
        setattr(cfg_abl, k, v)

    anc_loader_abl = None
    if getattr(cfg_abl, "w_kd_local", 0) > 0:
        anc_cls_abl = find_knn_retain_classes(
            old_forget_proto, all_prototypes,
            split["forget_ids"], cfg_abl.k_anchor)
        anc_loader_abl = build_local_anchor_loader(
            eval_base, split["retain_train_idx"],
            anc_cls_abl, cfg.batch_size, cfg.num_workers)

    fisher_abl = fisher if cfg_abl.use_ewc else None
    theta_abl  = theta_star if cfg_abl.use_ewc else None

    model_abl = run_lpeu_v7_unlearning(
        model              = clone_model(original_model),
        old_model          = clone_model(original_model),
        forget_loader      = forget_train_loader,
        retain_loader      = retain_train_loader,
        retain_test_loader = retain_test_loader,
        anchor_loader      = anc_loader_abl,
        forget_ids         = split["forget_ids"],
        num_classes        = num_classes,
        cfg                = cfg_abl,
        retain_acc_ref     = retain_acc_ref,
        fisher             = fisher_abl,
        theta_star         = theta_abl,
        retain_prototypes  = all_prototypes,
    )
    return full_evaluate(
        f"Ablation:{name}", model_abl,
        original_model, retain_test_loader, forget_eval_loader,
        local_neighbor_loader, old_forget_proto, cfg_abl, num_classes,
    )


# Uncomment to run ablations:
# ablation_results = {}
# ablation_results["NoStaging"]   = run_ablation("NoStaging",  {"stage1_epochs": 0, "stage2_epochs": 0, "stage3_epochs": 20})
# ablation_results["NoEWC"]       = run_ablation("NoEWC",      {"use_ewc": False})
# ablation_results["NoPCGrad"]    = run_ablation("NoPCGrad",   {"use_pcgrad": False, "pcgrad_boost": 1.0})
# ablation_results["NoAnchor"]    = run_ablation("NoAnchor",   {"w_kd_local": 0.0})
# ablation_results["MarginZero"]  = run_ablation("MarginZero", {"repel_margin": 0.0})
# ablation_results["NoRepulsion"] = run_ablation("NoRepulsion",{"w_repel": 0.0})
# with open(os.path.join(cfg.output_dir, "ablation_v7.json"), "w") as f:
#     json.dump(ablation_results, f, indent=2)

In [23]:
# ===========================================================================
# ▓  CELL N: TUNING GUIDE
# ===========================================================================
"""
HOW TO READ V7 EPOCH LOGS:

Log format:
  [v7][03/20][ERASE]  erase=0.721  repel=0.843  anc=0.0000  ewc=0.0231
                      proto_sim=0.601(drop:+0.150)  H=5.61/5.70
                      fgt_acc=0.0000  ret_acc=0.0048  conf=3.1  boost=100×  anchor_w=0.0

KEY SIGNALS:
  Phase ERASE (epoch 0-4):
    → anc SHOULD be 0.0000 (anchor is off)
    → ewc SHOULD be > 0 (EWC protecting neck)
    → proto_sim SHOULD DROP each epoch (this is the main signal)
    → repel SHOULD start high (~0.9) and decrease as cluster scatters
    → If proto_sim NOT dropping → EWC too strong (lower w_ewc) OR
                                   boost too weak (raise pcgrad_boost to 150)

  Phase REFINE (epoch 5-14):
    → anc SHOULD appear (> 0)
    → proto_sim SHOULD continue dropping
    → If neighbour_shift > 0.30 at this phase → raise w_kd_local in stage2

  Phase STABLE (epoch 15-19):
    → proto_sim should have reached < 0.30 by now
    → If not → raise stage2_epochs to 15, lower stage3_epochs to 5

QUICK TUNING DECISION TREE:
  proto_sim NOT dropping in Phase 1
    → raise pcgrad_boost (100→150→200)
    → OR lower w_ewc (5.0→2.0→0.5)
    → OR raise w_repel (12→16) or w_erase (8→12)

  proto_sim drops in Phase 1 but REBOUNDS in Phase 2
    → anchor phase starts too strong → lower w_kd_local * 0.5 from 8.0 to 5.0

  ret_acc drops below 80% in Phase 1
    → raise w_ewc (5→10→20) to protect retain-critical weights
    → or raise retain_acc_floor to trigger earlier safety stop

  intra_cluster_sim starts at 0.90+
    → DELETE original_model.pt — it's a BN-Neck v6 model
    → Make sure you're using v7 FaceModel (Simple Neck)

  neighbor_shift stays > 0.3 at end
    → Raise w_kd_local to 20 in Phase 3 (stage3 full weight)

  cluster_sim_drop < 0.10
    → Raise w_repel in Phase 1 (12→16)
    → Raise repel_margin target (-0.3 → -0.5, more aggressive)
"""

print("\n[LPEU-v7] All cells ready.")
print(f"  Expected improvements over v6:")
print(f"  intra_cluster_sim  : 0.905 → 0.350   [Simple Neck fix]")
print(f"  forget_sim_drop    : +0.000 → +0.250+ [Staged unlearn]")
print(f"  cluster_sim_drop   : +0.000 → +0.200+ [Aggressive repel]")
print(f"  neighbor_shift     : 0.165 → < 0.150  [Phase3 anchor]")
print(f"  MIA AUC            : 0.500 → < 0.530  [Better cluster erasure]")
print(f"\n  ⚠ IMPORTANT: Delete old model cache if exists:")
print(f"     import os; os.remove('/kaggle/working/runs/lpeu_v7/original_model.pt')")
print(f"     (First run will train from scratch — takes ~10 min on GPU)")


[LPEU-v7] All cells ready.
  Expected improvements over v6:
  intra_cluster_sim  : 0.905 → 0.350   [Simple Neck fix]
  forget_sim_drop    : +0.000 → +0.250+ [Staged unlearn]
  cluster_sim_drop   : +0.000 → +0.200+ [Aggressive repel]
  neighbor_shift     : 0.165 → < 0.150  [Phase3 anchor]
  MIA AUC            : 0.500 → < 0.530  [Better cluster erasure]

  ⚠ IMPORTANT: Delete old model cache if exists:
     import os; os.remove('/kaggle/working/runs/lpeu_v7/original_model.pt')
     (First run will train from scratch — takes ~10 min on GPU)
